In [0]:
# Databricks notebook source
# =============================================================
# OpsIntel Copilot
# Week 7 — Sync Gold Tables to RDS PostgreSQL
#
# Purpose:
# Reads all 4 gold Delta tables from Databricks
# and upserts data into RDS PostgreSQL so FastAPI
# can serve it with low latency
#
# Tables synced:
# gold_security_alerts    → security_alerts
# gold_correlation_records → correlations
# gold_incident_summary   → incident_summary
# gold_data_quality_results → data_quality_results
# =============================================================

import boto3
import json
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime

CATALOG = "workspace"
SCHEMA  = "opsintel_copilot"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("✅ Imports done")

In [0]:
# RDS connection details — hardcoded for this internal sync notebook only
# This notebook is never committed to GitHub with real values
# FastAPI (production-facing) uses Secrets Manager properly

RDS_HOST     = "opsintel-copilot-db.cmxb3jmq36ke.us-east-1.rds.amazonaws.com"
RDS_PORT     = 5432
RDS_DB       = "opsintel"
RDS_USER     = "opsintel_admin"
RDS_PASSWORD = "OpsIntel2024Secure!"

print("✅ RDS credentials set")
print(f"   Host: {RDS_HOST}")
print(f"   Database: {RDS_DB}")

In [0]:
def get_rds_connection():
    return psycopg2.connect(
        host=RDS_HOST,
        port=RDS_PORT,
        database=RDS_DB,
        user=RDS_USER,
        password=RDS_PASSWORD,
        connect_timeout=10
    )

# Test connection
conn = get_rds_connection()
conn.close()
print("✅ RDS connection successful")

In [0]:
def sync_security_alerts():
    print("\n🔄 Syncing gold_security_alerts → RDS security_alerts...")
    
    df = spark.sql("""
        SELECT
            event_id,
            event_type,
            user_email,
            source_ip,
            region,
            event_time,
            success,
            is_brute_force,
            is_escalation,
            is_impossible_travel,
            is_large_export,
            is_token_rotation,
            alert_generated_at
        FROM workspace.opsintel_copilot.gold_security_alerts
    """)
    
    rows = df.collect()
    print(f"   Rows to sync: {len(rows)}")
    
    if len(rows) == 0:
        print("   ⚠️ No rows to sync")
        return 0
    
    conn = get_rds_connection()
    cur = conn.cursor()
    
    cur.execute("TRUNCATE TABLE security_alerts")
    
    data = [(
        row["event_id"],
        row["event_type"],
        row["user_email"],
        row["source_ip"],
        row["region"],
        row["event_time"],
        row["success"],
        row["is_brute_force"],
        row["is_escalation"],
        row["is_impossible_travel"],
        row["is_large_export"],
        row["is_token_rotation"],
        row["alert_generated_at"]
    ) for row in rows]
    
    execute_values(cur, """
        INSERT INTO security_alerts (
            event_id, event_type, user_email, source_ip, region,
            event_time, success, is_brute_force, is_escalation,
            is_impossible_travel, is_large_export, is_token_rotation,
            alert_generated_at
        ) VALUES %s
    """, data)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"   ✅ Synced {len(rows)} rows to security_alerts")
    return len(rows)

sync_security_alerts()

In [0]:
def sync_correlations():
    print("\n🔄 Syncing gold_correlation_records → RDS correlations...")
    df = spark.sql("""
        SELECT
            correlation_id,
            correlation_type,
            pipeline_run_id,
            pipeline_name,
            pipeline_failed_at,
            error_message,
            security_event_id,
            security_event_type,
            security_event_at,
            involved_user,
            time_diff_minutes,
            confidence_score,
            recommendation,
            correlated_at
        FROM workspace.opsintel_copilot.gold_correlation_records
    """)
    rows = df.collect()
    print(f"   Rows to sync: {len(rows)}")
    if len(rows) == 0:
        print("   ⚠️ No rows to sync")
        return 0
    conn = get_rds_connection()
    cur = conn.cursor()
    cur.execute("TRUNCATE TABLE correlations")
    data = [(
        row["correlation_id"],
        row["correlation_type"],
        row["pipeline_run_id"],
        row["pipeline_name"],
        row["pipeline_failed_at"],
        row["error_message"],
        row["security_event_id"],
        row["security_event_type"],
        row["security_event_at"],
        row["involved_user"],
        float(row["time_diff_minutes"]) if row["time_diff_minutes"] is not None else None,
        float(row["confidence_score"]) if row["confidence_score"] is not None else None,
        row["recommendation"],
        row["correlated_at"]
    ) for row in rows]
    execute_values(cur, """
        INSERT INTO correlations (
            correlation_id, correlation_type, pipeline_run_id, pipeline_name,
            pipeline_failed_at, error_message, security_event_id, security_event_type,
            security_event_at, involved_user, time_diff_minutes, confidence_score,
            recommendation, correlated_at
        ) VALUES %s
    """, data)
    conn.commit()
    cur.close()
    conn.close()
    print(f"   ✅ Synced {len(rows)} rows to correlations")
    return len(rows)

sync_correlations()

In [0]:
def sync_incident_summary():
    print("\n🔄 Syncing gold_incident_summary → RDS incident_summary...")
    
    df = spark.sql("""
        SELECT
            incident_hour,
            region,
            total_events,
            brute_force_count,
            escalation_count,
            impossible_travel_count,
            large_export_count,
            summary_generated_at
        FROM workspace.opsintel_copilot.gold_incident_summary
    """)
    
    rows = df.collect()
    print(f"   Rows to sync: {len(rows)}")
    
    if len(rows) == 0:
        print("   ⚠️ No rows to sync")
        return 0
    
    conn = get_rds_connection()
    cur = conn.cursor()
    
    cur.execute("TRUNCATE TABLE incident_summary")
    
    data = [(
        row["incident_hour"],
        row["region"],
        row["total_events"],
        row["brute_force_count"],
        row["escalation_count"],
        row["impossible_travel_count"],
        row["large_export_count"],
        row["summary_generated_at"]
    ) for row in rows]
    
    execute_values(cur, """
        INSERT INTO incident_summary (
            incident_hour, region, total_events, brute_force_count,
            escalation_count, impossible_travel_count, large_export_count,
            summary_generated_at
        ) VALUES %s
    """, data)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"   ✅ Synced {len(rows)} rows to incident_summary")
    return len(rows)

sync_incident_summary()

In [0]:
def sync_data_quality():
    print("\n🔄 Syncing gold_data_quality_results → RDS data_quality_results...")
    
    df = spark.sql("""
        SELECT
            order_id,
            amount,
            currency,
            created_at,
            quality_flag,
            checked_at
        FROM workspace.opsintel_copilot.gold_data_quality_results
    """)
    
    rows = df.collect()
    print(f"   Rows to sync: {len(rows)}")
    
    if len(rows) == 0:
        print("   ⚠️ No rows to sync")
        return 0
    
    conn = get_rds_connection()
    cur = conn.cursor()
    
    cur.execute("TRUNCATE TABLE data_quality_results")
    
    data = [(
        row["order_id"],
        float(row["amount"]) if row["amount"] else None,
        row["currency"],
        row["created_at"],
        row["quality_flag"],
        row["checked_at"]
    ) for row in rows]
    
    execute_values(cur, """
        INSERT INTO data_quality_results (
            order_id, amount, currency, created_at, quality_flag, checked_at
        ) VALUES %s
    """, data)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"   ✅ Synced {len(rows)} rows to data_quality_results")
    return len(rows)

sync_data_quality()

In [0]:
print("\n" + "="*60)
print("✅ RDS SYNC COMPLETE")
print("="*60)

conn = get_rds_connection()
cur = conn.cursor()

tables = ["security_alerts", "correlations", "incident_summary", "data_quality_results"]

for table in tables:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"   {table}: {count} rows")

cur.close()
conn.close()
print("="*60)